In [ ]:
import serial
import serial.tools.list_ports
import time
from IPython.display import clear_output

def find_working_bluetooth_port():
    """Loops through all COM ports, ignores the physical USB cable, and finds the wireless one."""
    ports = list(serial.tools.list_ports.comports())
    print(f"Detected {len(ports)} total ports. Testing for the wireless link...")
    
    # Test ports in reverse order
    for p in reversed(ports):
        # HARD STOP: Skip your physical USB wire so we don't accidentally connect to it
        if "com9" in p.device.lower() or "usb" in p.description.lower():
            print(f"Skipping {p.device} (This is your physical USB wire)...")
            continue
            
        print(f"Testing {p.device} ({p.description})...")
        try:
            # Try to open the port with a short timeout to skip broken ones quickly
            test_ser = serial.Serial(p.device, baudrate=115200, timeout=0.5)
            
            # Flush and wait a split second to see if data is waiting
            test_ser.reset_input_buffer()
            time.sleep(0.2)
            
            if test_ser.in_waiting > 0:
                print(f"-> Success! Found active data stream on {p.device}")
                return p.device
            
            # Verify it's open and responsive
            test_ser.readline()
            test_ser.close()
            return p.device
            
        except Exception:
            print(f"   Skipping {p.device}: Port unavailable or locked.")
            continue
            
    return None

# 1. Automatically find the working wireless port
com_port = find_working_bluetooth_port()

if not com_port:
    print("\nError: Could not find the wireless port. Is the Feather paired in Windows Bluetooth settings?")
else:
    print(f"\nTarget locked on WIRELESS port {com_port}. Initializing live parsing pipeline...")    
    try:
        # 2. Open the confirmed working port
        ser = serial.Serial(com_port, baudrate=115200, timeout=1)
        ser.reset_input_buffer() 
        time.sleep(0.1)
        print("Live data stream active. Press the Jupyter STOP button to halt.")
        
        # 3. Your exact, untouched read and parsing loop
        while True:
            try:
                line = ser.readline().decode('utf-8').strip()
                if not line:
                    continue

                clean_line = line.replace('(', '').replace(')', '')
                parts = clean_line.split(',')
                
                if len(parts) == 3:
                    data_dict = {
                        'x': float(parts[0].strip()),
                        'y': float(parts[1].strip()),
                        'z': float(parts[2].strip())
                    }
                    
                    clear_output(wait=True)
                    print(data_dict)
                    
            except ValueError:
                continue
                
    except KeyboardInterrupt:
        print("\nStream safely paused by user.")
    except Exception as e:
        print(f"\nConnection error: {e}")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
            print("Serial pipeline cleanly released.")
